In [2]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from transformers import ASTForAudioClassification
from transformers import DefaultDataCollator
#from datasets import load_metric
import evaluate
from transformers import Trainer, TrainingArguments
import os

In [2]:

#spec = np.load("path_to_spec.npy")  # shape: (128, time)
path_spec = Path(r"C:\Polina\master\thesis\beat_this\data\audio\spectograms_npz\gtzan.npz")
data = np.load(path_spec)
lst = data.files
item = "gtzan_blues_00099/track"
# for item in lst:
#     print(item)
spectrogram = data[item].transpose()
#spec = spec[None, :, :]
print(f"spectrogram shape before {spectrogram.shape}")
spec = spectrogram[None, :, :]
print(f"spectrogram shape after {spec.shape}")


spectrogram shape before (128, 1501)
spectrogram shape after (1, 128, 1501)


In [5]:
path = os.path.join(Path(r"C:\Polina\master\thesis\beat_this\data\audio\spectrograms\gtzan_old\gtzan_blues_00000"), "track.npy")
spec_ex =  np.load(path)

In [ ]:
data = Path(r"C:\Polina\master\thesis\beat_this\data\audio\spectrograms\gtzan_old")
for file in os.listdir(data):
    path_file = os.path.join(data, file, "track.npy")
    print(file[6:][:-6])

In [49]:
spec_example = train_dataset[0]["input_values"]
print(spec_example.shape)
# #print(spec_example.squeeze(0).shape)
# max_time = 1000
# spec_example = le[:, :max_time]
# print(spec_example.shape)

torch.Size([96, 128])


In [38]:
# dataset =[]
# labels = []
# # for file in data:
# #     spectrogram = data[file].transpose()
# #     #spec = spectrogram[None, :, :]
# #     labels.append(file[6:][:-12])
# #     dataset.append(spec)
tracks_path = []
labels = []
data_path = Path(r"C:\Polina\master\thesis\beat_this\data\audio\spectrograms\gtzan_old")
for file in os.listdir(data_path):
    path_file = os.path.join(data, file, "track.npy")
    tracks_path.append(path_file)
    labels.append(file[6:][:-6])
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
train, validation, train_labels, val_labels = train_test_split(
        tracks_path, encoded_labels, test_size=0.2, stratify=encoded_labels, random_state=42)



In [ ]:
class GTZANSpectrogramDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels
        self.max_time = 1020
        self.target_time_steps = 1214
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        #spec = self.spectrograms[idx]  # shape: (128, time)
        spec = np.load(self.paths[idx])
        #spec = spec[None, :, :]  # add channel dim
        if spec.ndim == 3 and spec.shape[0] == 1:
            spec = spec.squeeze(0)
        # if spec.shape[1] < self.target_time_steps:
        #     # Pad with zeros if shorter
        #     pad_width = ((0, 0), (0, self.target_time_steps - spec.shape[1]))
        #     spec = np.pad(spec, pad_width, mode='constant')
        # else:
        #     # Truncate if longer
        #     spec = spec[:, :self.target_time_steps]
        spec = spec[:self.max_time, :]
        spec = torch.tensor(spec, dtype=torch.float32)
        
        #spec = spec.unsqueeze(0)
        label = self.labels[idx]
        return {"input_values": spec, "labels": label}

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,  # GTZAN has 10 genres
    ignore_mismatched_sizes=True  # allows adjusting output layer
)

data_collator = DefaultDataCollator()
#metric = load_metric("accuracy")
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)
training_args = TrainingArguments(
    output_dir="./ast-gtzan",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    num_train_epochs=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to=None
)
train_dataset = GTZANSpectrogramDataset(train, train_labels)
val_dataset = GTZANSpectrogramDataset(validation, val_labels)
trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=None,  # AST doesn't use a tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/1000 [00:00<?, ?it/s]

In [31]:
dataset =[]
labels = []
for file in data:
    spectrogram = data[file].transpose()
    spec = spectrogram[None, :, :]
    labels.append(file[6:][:-12])
    dataset.append(spec)
#dataset = np.array(dataset)
#dataset.shape

In [35]:
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)

In [38]:
train, validation, train_labels, val_labels = train_test_split(
        dataset, encoded_labels, test_size=0.2, stratify=encoded_labels, random_state=42)

In [ ]:


class GTZANSpectrogramDataset(Dataset):
    def __init__(self, spectrogram_paths, labels):
        self.paths = spectrogram_paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        spec = np.load(self.paths[idx])  # shape: (128, time)
        spec = spec[None, :, :]  # add channel dim
        spec = torch.tensor(spec, dtype=torch.float32)
        label = self.labels[idx]
        return {"pixel_values": spec, "labels": label}

In [ ]:


model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,  # GTZAN has 10 genres
    ignore_mismatched_sizes=True  # allows adjusting output layer
)

config.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

c:\Polina\master\thesis\annotations\hugginggf\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\polin\.cache\huggingface\hub\models--MIT--ast-finetuned-audioset-10-10-0.4593. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fa

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

data_collator = DefaultDataCollator()
#metric = load_metric("accuracy")
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)


In [10]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./ast-gtzan",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    num_train_epochs=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
)

In [40]:
train_dataset = GTZANSpectrogramDataset(train, train_labels)
val_dataset = GTZANSpectrogramDataset(validation, val_labels)

In [41]:
trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=None,  # AST doesn't use a tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

c:\Polina\master\thesis\annotations\hugginggf\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [42]:
trainer.train()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: polinaitsme (polinaitsme-RWTH Aachen University) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  0%|          | 0/1000 [00:00<?, ?it/s]

TypeError: expected str, bytes or os.PathLike object, not ndarray